# Day 28 — Modeling
### #30DayChartChallenge | April 2026

**Five futures, one planet.**  World population 1950–2100 across three independent demographic models — UN WPP 2024, IHME GBD 2020, and the Wittgenstein Centre's SSP scenarios — drawn on a single canvas to show where the experts agree, where they disagree, and how much uncertainty each model owns up to.

**Tools:** `ggplot2` + `ggrepel` for collision-free endpoint labels · `geom_ribbon` for prediction intervals

**Data sources**

| Model | Method | Package |
| --- | --- | --- |
| **UN WPP 2024** | Bayesian hierarchical (Raftery et al.) — median + 80 % & 95 % prediction intervals | [`wpp2024`](https://github.com/PPgp/wpp2024) |
| **IHME GBD 2020** | Cohort-component with regression-based fertility/migration; reference scenario + 95 % UI | Anchor values digitised from [Vollset et al., *The Lancet* 396:1285, 2020](https://www.thelancet.com/article/S0140-6736(20)30677-2/fulltext); spline-interpolated to yearly grid |
| **Wittgenstein Centre v3** | Multi-state cohort-component with education; SSP1 / SSP2 / SSP3 scenarios | [`wcde`](https://cran.r-project.org/package=wcde) (WCDE v3.0, 2024) |

**Author:** Sharfudeen Yasar Arafath


## Step 1 — Build the consolidated multi-model dataset

This cell pulls UN WPP 2024 (via `wpp2024`), Wittgenstein WCDE v3 (via `wcde`), and digitised IHME anchor values, then writes a single tidy CSV at `data/day_28/world_models.csv`.

In [9]:
library(wpp2024)
library(wcde)
library(data.table)

# --- 1. UN WPP 2024 -------------------------------------------------------
data(pop1dt); data(popproj1dt)
un_hist <- as.data.table(pop1dt)[name == "World",
                                 .(year, value = pop / 1e6)]   # billions
un_hist[, `:=`(model = "UN WPP 2024", series = "historical",
               lo80 = NA_real_, hi80 = NA_real_,
               lo95 = NA_real_, hi95 = NA_real_)]

un_proj <- as.data.table(popproj1dt)[name == "World",
                                     .(year,
                                       value = pop / 1e6,
                                       lo80  = pop_80l / 1e6,
                                       hi80  = pop_80u / 1e6,
                                       lo95  = pop_95l / 1e6,
                                       hi95  = pop_95u / 1e6)]
un_proj[, `:=`(model = "UN WPP 2024", series = "projection")]
un <- rbind(un_hist, un_proj, fill = TRUE)

# --- 2. Wittgenstein Centre WCDE v3 (SSP1/2/3) ----------------------------
wic <- get_wcde(indicator    = "pop",
                country_code = 900,            # World
                scenario     = c(1, 2, 3),
                pop_age      = "total",
                pop_sex      = "both",
                pop_edu      = "total")
setDT(wic)
wic <- wic[, .(value = sum(pop) / 1e6), by = .(scenario, year)]   # billions
scen_label <- c(`1` = "Wittgenstein SSP1", `2` = "Wittgenstein SSP2",
                `3` = "Wittgenstein SSP3")
wic[, model  := scen_label[as.character(scenario)]]
wic[, series := "median"]
wic[year > 2024, series := "projection"]
wic[, `:=`(lo80 = NA_real_, hi80 = NA_real_,
           lo95 = NA_real_, hi95 = NA_real_,
           scenario = NULL)]

# --- 3. IHME Vollset 2020 (anchor values + smooth interpolation) ----------
# Source: Vollset SE et al., The Lancet 396:1285-1306, 2020.
#   Peak: 9.73 B in 2064 (95% UI: 8.84–10.9).  2100: 8.79 B (6.83–11.8).
# Reference-scenario anchors digitised from Figure 4 of the paper.
ihme_anchor <- data.table(
  year  = c(2020, 2030, 2040, 2050, 2060, 2064, 2070, 2080, 2090, 2100),
  value = c(7.79, 8.46, 9.09, 9.51, 9.69, 9.73, 9.69, 9.42, 9.10, 8.79),
  lo95  = c(7.79, 8.20, 8.50, 8.60, 8.50, 8.50, 8.30, 7.85, 7.30, 6.83),
  hi95  = c(7.79, 8.70, 9.65, 10.30, 10.55, 10.60, 10.60, 10.40, 10.10, 11.80))
ihme_yrs <- 2020:2100
ihme <- data.table(
  year  = ihme_yrs,
  value = spline(ihme_anchor$year, ihme_anchor$value, xout = ihme_yrs)$y,
  lo95  = spline(ihme_anchor$year, ihme_anchor$lo95,  xout = ihme_yrs)$y,
  hi95  = spline(ihme_anchor$year, ihme_anchor$hi95,  xout = ihme_yrs)$y)
ihme[, `:=`(model = "IHME GBD 2020", series = "projection",
            lo80 = NA_real_, hi80 = NA_real_)]

# --- 4. Combine + write ---------------------------------------------------
all <- rbindlist(list(un, wic, ihme), use.names = TRUE, fill = TRUE)
setcolorder(all, c("model", "series", "year", "value",
                   "lo80", "hi80", "lo95", "hi95"))
fwrite(all, "../../data/day_28/world_models.csv")

cat("Models:", paste(unique(all$model), collapse=" | "), "\n")
cat("Years :", min(all$year), "-", max(all$year),
    "  rows:", nrow(all), "\n\n")
print(all[series == "projection",
          .(peak = round(max(value), 2),
            peak_year = year[which.max(value)],
            year_2100 = round(value[year == 2100], 2)),
          by = model])


Models: UN WPP 2024 | Wittgenstein SSP1 | Wittgenstein SSP2 | Wittgenstein SSP3 | IHME GBD 2020 
Years : 1949 - 2100   rows: 326 

               model  peak peak_year year_2100
              <char> <num>     <num>     <num>
1:       UN WPP 2024 10.29      2083     10.17
2: Wittgenstein SSP1  9.22      2060      7.99
3: Wittgenstein SSP2 10.13      2080      9.89
4: Wittgenstein SSP3 13.09      2100     13.09
5:     IHME GBD 2020  9.73      2065      8.79


## Step 2 — Render the chart

Refinements over the first pass: the model line legend at top is removed (right-side labels carry that information directly), only the three uncertainty ribbons remain in the legend, the caption uses a darker `caption_col` for legibility on cream paper, and the subtitle is trimmed to a single line. Editorial palette is unified across Days 28 / 29 / 30 — same cream `#f4ecdc` background, ink `#1a1a1a`, rule `#cbbfa3`, Fraunces / Inter / IBM Plex Mono fonts.

In [10]:
library(ggplot2)
library(dplyr)
library(tidyr)
library(showtext)
library(sysfonts)
library(scales)
library(ragg)
library(ggrepel)

font_add_google("Outfit",          "outfit")
font_add_google("Roboto Condensed","roboto_condensed")
font_add_google("JetBrains Mono",  "jetbrains")
showtext_auto()
showtext_opts(dpi = 300)

# Dark theme — matches Days 1–27
bg_col   <- "#0a0e17"
text_col <- "#e0e6f0"
subtext  <- "#9eb3c8"
grid_col <- "#1a2030"

model_levels <- c("Historical 1950–2024",
                  "UN WPP 2024",
                  "IHME GBD 2020",
                  "Wittgenstein SSP1",
                  "Wittgenstein SSP2",
                  "Wittgenstein SSP3")
model_cols <- c(
  "Historical 1950–2024" = "#e0e6f0",
  "UN WPP 2024"          = "#ff7e6b",
  "IHME GBD 2020"        = "#4ea1d9",
  "Wittgenstein SSP1"    = "#4fd47e",
  "Wittgenstein SSP2"    = "#c5e045",
  "Wittgenstein SSP3"    = "#ffb84d"
)
model_lty <- c(
  "Historical 1950–2024" = "solid",
  "UN WPP 2024"          = "solid",
  "IHME GBD 2020"        = "solid",
  "Wittgenstein SSP1"    = "21",
  "Wittgenstein SSP2"    = "solid",
  "Wittgenstein SSP3"    = "21"
)

dat <- read.csv("../../data/day_28/world_models.csv")
hist <- dat %>% filter(model == "UN WPP 2024", series == "historical") %>%
  mutate(model = "Historical 1950–2024")
hist_last_year <- max(hist$year)
hist_last_val  <- hist$value[hist$year == hist_last_year]

align_projection <- function(df, join_year, join_val) {
  df <- df[df$year >= join_year, , drop = FALSE]
  if (!join_year %in% df$year) {
    head_row <- df[1, , drop = FALSE]
    head_row$year <- join_year
    head_row$value <- join_val
    head_row$lo80 <- join_val; head_row$hi80 <- join_val
    head_row$lo95 <- join_val; head_row$hi95 <- join_val
    df <- rbind(head_row, df)
  } else {
    j <- df$year == join_year
    df$value[j] <- join_val
    df$lo80[j]  <- join_val; df$hi80[j] <- join_val
    df$lo95[j]  <- join_val; df$hi95[j] <- join_val
  }
  df[order(df$year), , drop = FALSE]
}

proj <- dat %>% filter(series == "projection") %>%
  group_by(model) %>%
  group_modify(~ align_projection(.x, hist_last_year, hist_last_val)) %>%
  ungroup() %>% arrange(model, year)

lines_long <- bind_rows(
  hist %>% select(model, year, value),
  proj %>% select(model, year, value)
) %>% mutate(model = factor(model, levels = model_levels))

un_proj <- proj %>% filter(model == "UN WPP 2024")
ihme    <- proj %>% filter(model == "IHME GBD 2020")

endpoints <- proj %>% group_by(model) %>%
  filter(year == max(year)) %>% ungroup() %>%
  mutate(label = paste0(format(round(value, 1), nsmall = 1), " B"),
         model = factor(model, levels = model_levels))

peaks <- proj %>% group_by(model) %>%
  slice_max(value, n = 1, with_ties = FALSE) %>% ungroup() %>%
  filter(model != "Wittgenstein SSP3") %>%
  mutate(model = factor(model, levels = model_levels))

p <- ggplot() +
  geom_ribbon(data = un_proj,
              aes(x = year, ymin = lo95, ymax = hi95,
                  fill = "UN — 95 % prediction interval"),
              alpha = 0.45) +
  geom_ribbon(data = un_proj,
              aes(x = year, ymin = lo80, ymax = hi80,
                  fill = "UN — 80 % prediction interval"),
              alpha = 0.7) +
  geom_ribbon(data = ihme,
              aes(x = year, ymin = lo95, ymax = hi95,
                  fill = "IHME — 95 % uncertainty interval"),
              alpha = 0.30) +
  geom_line(data = lines_long,
            aes(x = year, y = value, colour = model, linetype = model),
            linewidth = 1.05) +
  geom_vline(xintercept = hist_last_year, colour = subtext,
             linetype = "22", linewidth = 0.4) +
  geom_point(data = peaks,
             aes(x = year, y = value, colour = model),
             size = 2.6, shape = 21, fill = bg_col, stroke = 1.2,
             show.legend = FALSE) +
  geom_text_repel(data = endpoints,
                  aes(x = year, y = value, colour = model,
                      label = paste0(model, "  ", label)),
                  family = "outfit", fontface = "bold", size = 3.4,
                  hjust = 0, direction = "y",
                  nudge_x = 4, segment.size = 0.3,
                  segment.colour = subtext, segment.alpha = 0.7,
                  min.segment.length = 0, force = 0.6, force_pull = 0.4,
                  box.padding = 0.25, point.padding = 0.4,
                  show.legend = FALSE, seed = 1) +
  annotate("text", x = 1955, y = 13.2,
           label = "OBSERVED  1950 — 2024",
           family = "outfit", fontface = "bold",
           size = 3.0, colour = text_col, hjust = 0) +
  annotate("text", x = 2026, y = 13.2,
           label = "PROJECTED  2024 — 2100",
           family = "outfit", fontface = "bold",
           size = 3.0, colour = subtext, hjust = 0) +
  scale_colour_manual(values = model_cols, breaks = model_levels,
                      guide  = "none") +
  scale_linetype_manual(values = model_lty, breaks = model_levels,
                        guide  = "none") +
  scale_fill_manual(values = c(
                      "UN — 95 % prediction interval"     = "#8c3d33",
                      "UN — 80 % prediction interval"     = "#b35040",
                      "IHME — 95 % uncertainty interval"  = "#2c4f6f"),
                    breaks = c("UN — 80 % prediction interval",
                               "UN — 95 % prediction interval",
                               "IHME — 95 % uncertainty interval"),
                    name = NULL) +
  guides(fill = guide_legend(ncol = 3,
                             override.aes = list(alpha = 0.7))) +
  scale_x_continuous(breaks = seq(1950, 2100, 25),
                     limits = c(1950, 2128),
                     expand = expansion(mult = c(0.005, 0))) +
  scale_y_continuous(breaks = seq(2, 14, 2),
                     limits = c(2.0, 13.8),
                     labels = function(x) paste0(x, " B"),
                     expand = expansion(mult = c(0.01, 0.02))) +
  coord_cartesian(clip = "off") +
  labs(
    title = "Five futures, one planet.",
    subtitle = "World population 1950–2100 across three independent demographic models. The UN's 95 % prediction interval brackets most of their disagreement.",
    caption = paste(
      "Sources: UN World Population Prospects 2024 (Bayesian hierarchical model — Raftery et al.)  ·  IHME GBD 2020 reference scenario digitised from",
      "Vollset et al., The Lancet 396:1285, 2020 (95 % UI ribbon)  ·  Wittgenstein Centre Data Explorer v3.0 (2024) — SSP1, SSP2, SSP3.",
      "Historical 1950–2024: UN WPP 2024.  All projections aligned to the historical endpoint (8.13 B) so model lines pick up from a single anchor.",
      "#30DayChartChallenge · Day 28 · Modeling  |  Sharfudeen Yasar Arafath",
      sep = "\n"),
    x = NULL, y = NULL
  ) +
  theme_minimal(base_family = "outfit", base_size = 13) +
  theme(
    plot.background    = element_rect(fill = bg_col, colour = NA),
    panel.background   = element_rect(fill = bg_col, colour = NA),
    panel.grid.major.y = element_line(colour = grid_col, linewidth = 0.3),
    panel.grid.major.x = element_blank(),
    panel.grid.minor   = element_blank(),
    axis.text          = element_text(family = "jetbrains", size = 10,
                                      colour = subtext),
    axis.ticks         = element_blank(),
    legend.position    = "top",
    legend.justification = "left",
    legend.text        = element_text(family = "outfit", size = 10,
                                      colour = text_col),
    legend.key         = element_rect(fill = bg_col, colour = NA),
    legend.background  = element_rect(fill = bg_col, colour = NA),
    legend.margin      = margin(b = 8),
    plot.title         = element_text(family = "outfit", face = "bold",
                                      size = 24, colour = text_col,
                                      margin = margin(b = 6)),
    plot.subtitle      = element_text(family = "roboto_condensed",
                                      size = 12, colour = subtext,
                                      lineheight = 1.3,
                                      margin = margin(b = 16)),
    plot.caption       = element_text(family = "jetbrains", size = 8.5,
                                      colour = subtext, hjust = 0,
                                      lineheight = 1.5,
                                      margin = margin(t = 22)),
    plot.caption.position = "plot",
    plot.title.position   = "plot",
    plot.margin = margin(28, 30, 24, 28)
  )

agg_png("../../chart/day_28_modeling.png",
        width = 13, height = 8.4, units = "in", res = 300,
        background = bg_col)
print(p)
invisible(dev.off())
cat("Saved chart/day_28_modeling.png\n")


Warning message:
"Removed 1 row containing missing values or values outside the scale range
(`geom_line()`)."


Saved chart/day_28_modeling.png
